# NYC Urban Analytics Platform
### Аналитическая платформа на Microsoft Azure

## 📋 Обзор проекта
Проект строит unified аналитическую платформу на Azure, которая интегрирует 
три гетерогенных источника данных в единое хранилище с medallion архитектурой.

**Цель:** исследовать взаимосвязь между городской мобильностью, 
качеством воздуха и макроэкономическими показателями Нью-Йорка.

**Период:** 2024 год  
**Масштаб:** 41+ миллион записей  
**Стек:** Azure Data Lake Gen2, Azure Data Factory, Azure Databricks, PySpark

## 🏗️ Архитектура — Medallion

### Bronze (сырые данные)
| Источник | Файл | Размер |
|---|---|---|
| NYC Taxi TLC | yellow_tripdata_2024-01..12.parquet | ~700MB |
| OpenAQ API | openaq_nyc_2024.json | ~2MB |
| World Bank | worldbank_gdp_usa.json | <1MB |
| ECB | ecb_fx_usdeur.csv | <1MB |

### Silver (очищенные данные)
| Таблица | Записей | Что сделано |
|---|---|---|
| silver/taxi | 35.6М | Удалены нулевые пассажиры, аномальные поездки, добавлены pickup_date/hour/duration |
| silver/air_quality | 17,432 | Фильтрация по NYC bbox, категоризация PM2.5 |
| silver/economy/fx | 602 | Только 2024, выбраны date и rate |
| silver/economy/gdp | 5 | Только 2020-2024 |

### Gold (аналитические таблицы)
| Таблица | Записей | Описание |
|---|---|---|
| gold/taxi_daily | 80,594 | Агрегация по дням и зонам |
| gold/airquality_daily | 366 | Среднее PM2.5 по всему NYC |
| gold/mobility_vs_airquality | 366 | Объединённая таблица для корреляций |

## 📊 Источники данных

### 1. NYC Taxi (TLC)
- **URL:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- **Формат:** Parquet
- **Загрузка:** Azure Data Factory Pipeline с ForEach по месяцам
- **Ключевые поля:** pickup/dropoff datetime, fare_amount, trip_distance, 
  passenger_count, PULocationID, DOLocationID

### 2. OpenAQ (качество воздуха)
- **URL:** https://api.openaq.org/v3
- **Формат:** JSON API (требует X-API-Key)
- **Загрузка:** Python requests в Databricks
- **Покрытие:** 56 PM2.5 сенсоров по всему NYC (Манхэттен, Бронкс, Квинс, Бруклин)
- **Ключевые поля:** sensor_id, location_name, date, pm25_avg

### 3. World Bank GDP
- **URL:** https://api.worldbank.org/v2/country/USA/indicator/NY.GDP.MKTP.CD
- **Формат:** JSON API
- **Загрузка:** Python requests в Databricks
- **Ключевые поля:** year, gdp_usd

### 4. ECB FX (курс USD/EUR)
- **URL:** https://data-api.ecb.europa.eu/service/data/EXR/D.USD.EUR.SP00.A
- **Формат:** CSV
- **Загрузка:** Azure Data Factory Pipeline
- **Ключевые поля:** date, usd_eur_rate

## 🔍 Ключевые инсайты

### 1. Корреляция такси и качества воздуха
- **Коэффициент Пирсона: -0.197** — слабая отрицательная корреляция
- Вывод: количество поездок такси **не влияет** на уровень PM2.5
- Загрязнение воздуха больше зависит от **сезона и погодных условий**
- Зима (янв-март) — PM2.5 выше; Лето (июнь-август) — воздух чище

### 2. Паттерны мобильности
- Чёткий **недельный цикл**: пики в будни (96-120K поездок), 
  провалы в выходные (60-70K)
- Праздники дают резкие провалы (Новый год: 67K поездок)
- **Рост к концу года**: ноябрь-декабрь заметно активнее января

### 3. Экономика
- Ежедневная выручка: **$2-4.5 млн** в зависимости от дня
- Курс USD/EUR в 2024 стабилен (~1.09-1.12)
- Разрыв USD vs EUR минимален — валютный риск низкий
- ВВП США 2024: **$28.7 триллионов**

### 4. Качество данных
- Из 41.1М сырых записей такси удалено **5.5М (13.4%)** аномалий
- OpenAQ: 60 аномальных измерений отфильтровано из 17,492

## ⚙️ Техническая документация

### Инфраструктура Azure
| Ресурс | Название | Регион |
|---|---|---|
| Resource Group | nyc-analytics-rg | UAE North |
| Storage Account (ADLS Gen2) | nycanalyticsdl | UAE North |
| Azure Data Factory | nyc-analytics-adf | UAE North |
| Azure Databricks | nyc-analytics-dbx | East US |

### ADF Pipelines
| Pipeline | Источник | Назначение |
|---|---|---|
| pl_bronze_nyctaxi | TLC CloudFront CDN | bronze/taxi/ |
| pl_bronze_openaq | OpenAQ API v3 | bronze/air_quality/ |
| pl_bronze_economy | World Bank + ECB | bronze/economy/ |

### Databricks Notebooks
| Notebook | Описание |
|---|---|
| silver_nyctaxi | Bronze → Silver трансформации + Gold агрегации + визуализации |
| project_documentation | Документация проекта |

### Стоимость проекта
| Ресурс | Стоимость |
|---|---|
| Storage Account | ~$0.01/месяц |
| Azure Data Factory | ~$0.50 |
| Azure Databricks | ~$1.50 |
| **Итого** | **~$2.00** |

### Качество данных — Data Dictionary
| Поле | Тип | Описание |
|---|---|---|
| pickup_date | date | Дата посадки |
| pickup_hour | int | Час посадки (0-23) |
| trip_duration_min | double | Длительность поездки в минутах |
| PULocationID | int | ID зоны посадки (TLC zones) |
| fare_amount | double | Базовый тариф в USD |
| total_amount | double | Итоговая сумма в USD |
| pm25_daily_avg | double | Среднее PM2.5 по NYC в µg/m³ |
| pm25_category | string | Good/Moderate/Unhealthy |
| usd_eur_rate | double | Курс USD/EUR от ECB |
| gdp_usd | double | ВВП США в USD |

## 🔄 Как воспроизвести проект

### Предварительные требования
- Azure подписка (Pay-As-You-Go)
- OpenAQ API ключ (бесплатно на explore.openaq.org)

### Шаги
1. Создать Resource Group `nyc-analytics-rg`
2. Создать Storage Account `nycanalyticsdl` с Hierarchical Namespace
3. Создать контейнеры: `bronze`, `silver`, `gold`
4. Создать Azure Data Factory `nyc-analytics-adf`
5. Настроить Linked Services: ADLS Gen2, HTTP (TLC, OpenAQ, WorldBank, ECB)
6. Запустить pipelines: pl_bronze_nyctaxi → pl_bronze_economy
7. Создать Databricks workspace `nyc-analytics-dbx`
8. Запустить notebook `silver_nyctaxi` последовательно все ячейки

## ⚠️ Ограничения и допущения

### Данные
- OpenAQ покрывает только **56 сенсоров** — не все районы NYC представлены
- World Bank публикует GDP с задержкой ~6 месяцев
- ECB не публикует курс в выходные и праздники (NULL в таблице)
- NYC Taxi данные содержат ~13% аномальных записей

### Архитектура
- Bronze слой для OpenAQ и GDP загружается напрямую через Databricks
  (обход ADF из-за сложной структуры JSON)
- Нет автоматического обновления данных (scheduled triggers не настроены)
- Row-Level Security в Power BI не реализован

### Аналитика
- Корреляция такси/PM2.5 считается на дневном уровне —
  почасовой анализ даст точнее результаты
- Не учитываются другие виды транспорта (метро, автобусы, Uber)

## 🚀 Дальнейшее развитие

### Данные
- Добавить данные Uber/Lyft для полной картины мобильности
- Интегрировать данные о погоде (температура, осадки, ветер)
- Добавить данные метро NYC для мультимодального анализа
- Расширить период до 3-5 лет для долгосрочных трендов

### Архитектура
- Настроить scheduled triggers в ADF для автообновления
- Реализовать Delta Lake вместо Parquet для ACID транзакций
- Добавить Data Quality checks через Great Expectations
- Настроить Azure Monitor для алертов о сбоях pipeline

### Аналитика
- Построить Power BI дашборды для бизнес-пользователей
- Добавить ML модель прогноза спроса на такси
- Реализовать почасовой анализ корреляции трафик/PM2.5
- Добавить геопространственный анализ по зонам NYC

%md## 📌 Выводы

Проект успешно демонстрирует:

✅ **Medallion архитектуру** — Bronze/Silver/Gold на Azure ADLS Gen2  
✅ **ETL пайплайны** — Azure Data Factory для оркестрации  
✅ **Big Data обработку** — PySpark на Databricks (41М+ записей)  
✅ **Кросс-доменную аналитику** — мобильность + экология + экономика  
✅ **Ключевой инсайт** — такси не влияет на PM2.5 (r = -0.197)  
✅ **Экономичность** — весь проект обошёлся ~$2

---
*Проект выполнен на Azure (аналог Microsoft Fabric Medallion Architecture)*  
*Автор: Artem Davydov | 2024*
